# Post-solve Jacobian & sensitivities from the held KKT factor (pounce#82)

`JaxProblem.batched_solve` is a differentiable, stacked block-diagonal
batched solve (pounce#76). Its `custom_vjp` backward already reuses the
IPM's converged FERAL compound KKT factor. This notebook shows the
**public, first-class** post-solve sensitivity surface built on that
held factor (pounce#82), so downstream code (e.g. a differentiable
projection layer) can:

- request the **full Jacobian** `J = ∂x*/∂p` directly, instead of
  wrapping `batched_solve` in `jax.jacrev`;
- **anchor** a solve once and reuse the held factor across several
  post-solve products without re-solving or risking LRU eviction;
- compute a **directional sensitivity** `Δx = J @ Δp` for a linear
  update *without* ever materialising `J`.

The API is four pieces:

| call | returns | mode |
|------|---------|------|
| `batched_solve_with_jacobian(p, x0)` | `x*, (lam, zL, zU), J` | full Jacobian |
| `anchor(p, x0)` → `AnchorState` | a held-factor handle | — |
| `batched_jvp_from_state(state, dp)` | `J @ dp`, shape `(B, n)` | forward |
| `batched_vjp_from_state(state, x_bar)` | `Jᵀ x̄`, shape `(B, p_dim)` | reverse |

All of them reuse the held LDLᵀ factor via one multi-RHS
`Solver.kkt_solve_many` — one Rust crossing, no NLP re-solve.

In [1]:
import jax
import jax.numpy as jnp
import numpy as np

from pounce.jax import JaxProblem

## 1. A small projection layer

Mirroring the `optlayer` use case: project a predicted point
$\hat y$ onto a context-dependent feasible set. The parameter packs
the prediction and the context, `p = [ŷ (ny), ctx (n_ctx)]`:

$$
x^*(p) = \arg\min_x \tfrac12\|x - \hat y\|^2 \quad\text{s.t.}\quad
\textstyle\sum_i x_i = \mathrm{ctx}_0,\;\; x_0 - x_1 \le \mathrm{ctx}_1,\;\;
0 \le x \le 1.
$$

Only the prediction block $\hat y = p_{:ny}$ carries the layer
Jacobian we care about — the context columns just shape the feasible
region.

In [2]:
ny, n_ctx = 3, 2


def f(x, p):
    y_hat = p[:ny]
    return jnp.sum((x - y_hat) ** 2)


def g(x, p):
    ctx = p[ny:]
    # equality: total mass = ctx[0];  inequality: x[0] - x[1] <= ctx[1]
    return jnp.stack([jnp.sum(x) - ctx[0], x[0] - x[1] - ctx[1]])


jp = JaxProblem(
    f=f, g=g, n=ny, m=2, p_example=jnp.zeros(ny + n_ctx),
    lb=jnp.zeros(ny), ub=jnp.ones(ny),
    cl=jnp.array([0.0, -1e19]), cu=jnp.array([0.0, 0.0]),
    options={"tol": 1e-10, "print_level": 0, "sb": "yes"},
)

# A minibatch of (y_hat, ctx) pairs.
p_batch = jnp.array([
    [0.6, 0.1, 0.5, 1.0, 0.3],
    [0.2, 0.7, 0.2, 1.0, 0.1],
])
x0 = jnp.full(ny, 1.0 / ny)

## 2. Full Jacobian in one call

`batched_solve_with_jacobian` returns the primal solution, the
`(lam, zL, zU)` duals (same contract as `batched_solve_with_warm`),
and the per-block Jacobian `J`. `wrt_cols=slice(0, ny)` keeps only the
prediction columns, so `J` is `(B, ny, ny)` instead of `(B, ny, p_dim)`.

The Jacobian is assembled from the held factor (one multi-RHS
back-solve over the identity output basis), so it agrees with
`jax.jacrev` over `batched_solve` to machine precision.

In [3]:
x_star, (lam, zL, zU), J = jp.batched_solve_with_jacobian(
    p_batch, x0, wrt_cols=slice(0, ny),
)

print("x_star\n", np.asarray(x_star))
print("\nJ (∂x*/∂ŷ) shape:", J.shape)
print(np.asarray(J))

# Cross-check against jax.jacrev over the full parameter, then slice the
# block-diagonal (∂x^(k)/∂p^(j) = 0 for k != j) and the prediction cols.
J_full = jax.jacrev(lambda P: jp.batched_solve(P, x0))(p_batch)
J_ref = jnp.stack([J_full[k, :, k, :ny] for k in range(p_batch.shape[0])])
print("\nmax |J - jacrev|:", float(jnp.max(jnp.abs(J - J_ref))))

x_star
 [[0.43333334 0.13333333 0.43333333]
 [0.16666667 0.66666667 0.16666667]]

J (∂x*/∂ŷ) shape: (2, 3, 3)
[[[ 0.16666667  0.16666667 -0.33333333]
  [ 0.16666667  0.16666667 -0.33333333]
  [-0.33333333 -0.33333333  0.66666667]]

 [[ 0.66666667 -0.33333333 -0.33333333]
  [-0.33333333  0.66666667 -0.33333333]
  [-0.33333333 -0.33333333  0.66666667]]]



max |J - jacrev|: 0.0


## 3. Anchor once, reuse the factor — use the context manager

For the linear-update pattern you solve once at an anchor point and
then apply several nearby sensitivity products against the *same* held
factor. `anchor(...)` pins that factor and hands back an `AnchorState`.

**Prefer the context manager** — it releases the factor
deterministically on block exit, even if an exception is raised:

In [4]:
# A small perturbation of the prediction block only.
dy = jnp.array([
    [0.05, -0.02, 0.0],
    [0.00,  0.03, -0.01],
])
x_bar = jnp.array([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
])

with jp.anchor(p_batch, x0, wrt_cols=slice(0, ny)) as state:
    # forward: directional sensitivity Δx = J @ Δŷ  (never builds J)
    dx = jp.batched_jvp_from_state(state, dy)
    # reverse: Δp_bar = Jᵀ @ x_bar
    dp_bar = jp.batched_vjp_from_state(state, x_bar)
    print("inside `with`, held factors:", len(jp._pinned_solvers))

# Factor is released on exit.
print("after `with`, held factors:", len(jp._pinned_solvers))
print("state.closed:", state.closed)

print("\nΔx = J @ Δŷ:\n", np.asarray(dx))
# The JVP equals the Jacobian contraction (machine precision) ...
print("max |Δx - J@Δŷ|:", float(jnp.max(jnp.abs(dx - jnp.einsum("knp,kp->kn", J, dy)))))

inside `with`, held factors: 1
after `with`, held factors: 0
state.closed: True

Δx = J @ Δŷ:
 [[ 0.005       0.005      -0.01      ]
 [-0.00666667  0.02333333 -0.01666667]]
max |Δx - J@Δŷ|: 4.336808689942018e-18


### Sanity check: JVP vs finite differences

`batched_solve` is a `custom_vjp`, so `jax.jvp` can't trace it; a
central finite-difference directional derivative confirms the sign and
magnitude of `J @ dp`. We pad the prediction perturbation with zeros in
the context columns to make a full-`p` direction.

In [5]:
dp_full = jnp.concatenate([dy, jnp.zeros((p_batch.shape[0], n_ctx))], axis=1)
eps = 1e-6
dx_fd = (jp.batched_solve(p_batch + eps * dp_full, x0)
         - jp.batched_solve(p_batch - eps * dp_full, x0)) / (2 * eps)
print("max |Δx - finite-difference|:", float(jnp.max(jnp.abs(dx - dx_fd))))

max |Δx - finite-difference|: 3.877048682099371e-11


## 4. Explicit lifetime across calls

Sometimes the anchor must outlive a single lexical block — e.g. the
handle is stored on a projection layer and reused across several
user-level calls before the next re-anchor. The same `AnchorState`
supports explicit ownership: keep it, reuse it, and `close()` when done.

Re-anchoring by overwriting the attribute *without* closing would leak
the held factor (the pinned registry keeps it alive). Use
`state.reanchor(...)`, which closes the prior pin before taking the new
one — the held-factor count stays at 1.

In [6]:
# Anchor and stash the handle (as a layer would).
state = jp.anchor(p_batch, x0, wrt_cols=slice(0, ny))
print("held factors after anchor:", len(jp._pinned_solvers))

# Several linear updates reuse the same factor.
for scale in (1.0, 2.0, 3.0):
    dx_s = jp.batched_jvp_from_state(state, scale * dy)
    print(f"  scale={scale}:  Δx[0] = {np.asarray(dx_s)[0]}")

# Move the anchor to a new operating point without leaking the old factor.
state.reanchor(p_batch.at[:, :ny].add(0.05), x0)
print("held factors after reanchor:", len(jp._pinned_solvers))

# Release when the chain is rejected / the layer is reset.
state.close()
print("held factors after close:", len(jp._pinned_solvers))

held factors after anchor: 1
  scale=1.0:  Δx[0] = [ 0.005  0.005 -0.01 ]
  scale=2.0:  Δx[0] = [ 0.01  0.01 -0.02]
  scale=3.0:  Δx[0] = [ 0.015  0.015 -0.03 ]
held factors after reanchor: 1
held factors after close: 0


## 5. Safety nets

The held factor is owned by the caller, so a missed `close()` would
leak it. Three guardrails make the unmanaged path safe-by-default:

- **`weakref` finalizer** — if a handle is garbage-collected without
  `close()`, the pin is still reclaimed.
- **Capacity cap** (`jp._pinned_capacity`, default 16) — too many live
  anchors raise loudly rather than growing unbounded.
- **Closed-handle guard** — a from-state call on a released handle
  raises a clear error instead of silently using a stale factor.

Still, **prefer the context manager**; reach for explicit `close()` /
`reanchor()` only when the handle genuinely has to span calls.

In [7]:
import gc

# Dropped without close() — the finalizer reclaims the factor on GC.
tmp = jp.anchor(p_batch, x0)
print("before drop:", len(jp._pinned_solvers))
del tmp
gc.collect()
print("after GC:   ", len(jp._pinned_solvers))

# Closed-handle guard.
st = jp.anchor(p_batch, x0)
st.close()
try:
    jp.batched_jvp_from_state(st, dy)
except RuntimeError as e:
    print("closed handle raises:", str(e).splitlines()[0])

before drop: 1


after GC:    0
closed handle raises: pounce.jax: AnchorState is closed; its held factor has been released. Re-anchor with jp.anchor(...) / jp.batched_solve_with_jacobian(..., return_state=True).


## When to reach for which

- **Training** — nothing changes: `jax.grad` / `jax.value_and_grad`
  through `batched_solve` already reuses the factor in its `custom_vjp`
  backward.
- **Validation / diagnostics** — `batched_solve_with_jacobian` when you
  want the explicit `J` (and the duals) as a first-class result.
- **Linear updates** — `anchor(...)` + `batched_jvp_from_state` when you
  only need `Δx = J @ Δp` for nearby perturbations and never want to
  pay for the full `J`. This is the cheapest path.
- **Custom reverse-mode plumbing** — `batched_vjp_from_state` exposes
  `Jᵀ x̄` against the held factor without going through `jax.vjp`.

All four share one held FERAL factor per anchor — one batched NLP solve,
one factorisation, many cheap multi-RHS sensitivity solves.